In [9]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd

df = pd.read_csv("trimmed-data-1.csv")

X = df.drop(columns=["Municipal", "Is Demolished"])
y = df["Municipal"]
X = pd.get_dummies(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBClassifier(random_state=42, scale_pos_weight=807/175)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.88      0.92      0.90       160
        True       0.55      0.43      0.48        37

    accuracy                           0.83       197
   macro avg       0.71      0.68      0.69       197
weighted avg       0.81      0.83      0.82       197



In [10]:
model = XGBClassifier(
    random_state=42,
    scale_pos_weight=807/175,
    max_depth=4,
    learning_rate=0.05,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.87      0.86      0.87       160
        True       0.44      0.46      0.45        37

    accuracy                           0.79       197
   macro avg       0.65      0.66      0.66       197
weighted avg       0.79      0.79      0.79       197



In [11]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred_adjusted = (y_proba >= 0.3).astype(bool)
print(classification_report(y_test, y_pred_adjusted))

              precision    recall  f1-score   support

       False       0.91      0.68      0.78       160
        True       0.34      0.70      0.46        37

    accuracy                           0.69       197
   macro avg       0.62      0.69      0.62       197
weighted avg       0.80      0.69      0.72       197



In [12]:
y_pred_adjusted = (y_proba >= 0.4).astype(bool)
print(classification_report(y_test, y_pred_adjusted))

              precision    recall  f1-score   support

       False       0.90      0.79      0.84       160
        True       0.41      0.62      0.49        37

    accuracy                           0.76       197
   macro avg       0.66      0.71      0.67       197
weighted avg       0.81      0.76      0.78       197



In [13]:
# filter to undesignated buildings
undesignated = df[df["Municipal"] == False].copy()

# prepare features the same way as training
X_undesignated = undesignated.drop(columns=["Municipal"])
X_undesignated = pd.get_dummies(X_undesignated)

# align columns with training data in case get_dummies created different columns
X_undesignated = X_undesignated.reindex(columns=X_train.columns, fill_value=0)

# get probabilities
undesignated["probability"] = model.predict_proba(X_undesignated)[:, 1]

# rank by probability
results = undesignated[["Name", "Community", "Year of Construction", "probability"]].sort_values("probability", ascending=False)
print(results.head(20))

                                 Name                 Community  \
255                 Court House No. 2  DOWNTOWN COMMERCIAL CORE   
215     Central Memorial Park Library                  BELTLINE   
24             7 Street NW Boulevards                  ROSEDALE   
464         Hotel Cecil (Cecil Hotel)     DOWNTOWN EAST VILLAGE   
116      Bow Valley Lawn Bowling Club                 HILLHURST   
881               Substation No. Four             SOUTH CALGARY   
59                Anderson Apartments         LOWER MOUNT ROYAL   
266                     Crescent Park          CRESCENT HEIGHTS   
760            Rotary Park Lawn Bowls          CRESCENT HEIGHTS   
562         Louise (Hillhurst) Bridge                 HILLHURST   
961                   Willingdon Hall               ASPEN WOODS   
57             Allen (Palace) Theatre  DOWNTOWN COMMERCIAL CORE   
820                   Smith Residence                ELBOW PARK   
451                  Hillhurst School                 HILLHURS